In [ ]:
#LAB Relevance Scoring and Rerankers for Trustworthy AI & EU AI Act
#Cindy Lund

In [1]:
#Imports and Config to install libraries
%pip install -U cohere langchain langchain-community langchain-openai langchain-cohere chromadb pypdf tiktoken


  Using cached fastavro-1.12.1-cp313-cp313-win_amd64.whl.metadata (5.9 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached types_requests-2.32.4.20260107-py3-none-any.whl.metadata (2.0 kB)
Using cached fastavro-1.12.1-cp313-cp313-win_amd64.whl (444 kB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached types_requests-2.32.4.20260107-py3-none-any.whl (20 kB)
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
    --------------------------------------- 0.5/21.9 MB 3.9 MB/s eta 0:00:06
   -- ------------------------------------- 1.6/21.9 MB 3.8 MB/s eta 0:00:06
   ---- ----------------------------------- 2.4/21.9 MB 3.8 MB/s eta 0:00:06
   ----- ---------------------------------- 3.1/21.9 MB 3.8 MB/s eta 0:00:05
   ------- -------------------------------- 3.9/21.9 MB 3.8 MB/s eta 0:00:05
   -------- ------------------------------- 4.7/21.9 MB 3.8 MB/s eta 0:00:05
   ---------- --------------------------

In [2]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads variables from .env into environment

print("OpenAI key loaded:", "OPENAI_API_KEY" in os.environ)
print("Cohere key loaded:", "COHERE_API_KEY" in os.environ)

OpenAI key loaded: True
Cohere key loaded: True


In [5]:
import shutil
print("ffmpeg found:", shutil.which("ffmpeg"))

ffmpeg found: C:\ProgramData\chocolatey\bin\ffmpeg.EXE


In [ ]:
#Split Audio Files into Chunks (too large to transcribe)
from pathlib import Path
import subprocess

AUDIO_DIR = Path("./data/audio")
CHUNKS_DIR = Path("./data/audio_chunks")
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

SEGMENT_TIME = 480  # 8 minutes

audio_files = sorted(AUDIO_DIR.glob("*.m4a"))
print("Found audio files:", [p.name for p in audio_files])
print("Chunks dir:", CHUNKS_DIR.resolve())

for audio_path in audio_files:
    out_pattern = CHUNKS_DIR / f"{audio_path.stem}_part_%03d.m4a"
    print(f"\nSplitting: {audio_path.name}")

    cmd = [
        "ffmpeg", "-hide_banner",
        "-i", str(audio_path),
        "-f", "segment",
        "-segment_time", str(SEGMENT_TIME),
        "-c", "copy",
        str(out_pattern)
    ]

    # capture output so we see errors in notebook
    result = subprocess.run(cmd, text=True, capture_output=True)
    print("Return code:", result.returncode)
    if result.stdout:
        print("STDOUT:", result.stdout[:1000])
    if result.stderr:
        print("STDERR:", result.stderr[:2000])

created = sorted(CHUNKS_DIR.glob("*.m4a"))
print("\n✅ Total chunks created:", len(created))
print("First 10 chunks:", [p.name for p in created[:10]])

Found audio files: ['Red_Lines_AI_Act.m4a', 'The_Blueprint_For_Trustworthy_AI.m4a']
Chunks dir: C:\Users\cindy\OneDrive\Documents\AI Ironhack Coursework\vscode101\WEEK03\LAB_Relevance_Scoring_Rerankers\data\audio_chunks

Splitting: Red_Lines_AI_Act.m4a
Return code: 0
STDERR: Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'data\audio\Red_Lines_AI_Act.m4a':
  Metadata:
    major_brand     : dash
    minor_version   : 0
    compatible_brands: iso6mp41
    creation_time   : 2026-01-24T10:34:04.000000Z
    encoder         : Google
  Duration: 00:16:02.49, start: 0.000000, bitrate: 257 kb/s
  Stream #0:0[0x1](und): Audio: aac (LC) (mp4a / 0x6134706D), 44100 Hz, stereo, fltp, 256 kb/s (default)
    Metadata:
      creation_time   : 2026-01-24T10:34:04.000000Z
      handler_name    : ISO Media file produced by Google Inc.
      vendor_id       : [0][0][0][0]
Stream mapping:
  Stream #0:0 -> #0:0 (copy)
[segment @ 0000012494b5ba00] Opening 'data\audio_chunks\Red_Lines_AI_Act_part_000.m4a' for writing


In [ ]:
#Confirm Chunks were created
from pathlib import Path

CHUNKS_DIR = Path("./data/audio_chunks")
chunks = sorted(CHUNKS_DIR.glob("*.m4a"))

print("Chunks dir exists:", CHUNKS_DIR.exists(), "->", CHUNKS_DIR.resolve())
print("Total chunk files:", len(chunks))
print("First 10:", [c.name for c in chunks[:10]])
print("Last 5:", [c.name for c in chunks[-5:]])

Chunks dir exists: True -> C:\Users\cindy\OneDrive\Documents\AI Ironhack Coursework\vscode101\WEEK03\LAB_Relevance_Scoring_Rerankers\data\audio_chunks
Total chunk files: 5
First 10: ['Red_Lines_AI_Act_part_000.m4a', 'Red_Lines_AI_Act_part_001.m4a', 'Red_Lines_AI_Act_part_002.m4a', 'The_Blueprint_For_Trustworthy_AI_part_000.m4a', 'The_Blueprint_For_Trustworthy_AI_part_001.m4a']
Last 5: ['Red_Lines_AI_Act_part_000.m4a', 'Red_Lines_AI_Act_part_001.m4a', 'Red_Lines_AI_Act_part_002.m4a', 'The_Blueprint_For_Trustworthy_AI_part_000.m4a', 'The_Blueprint_For_Trustworthy_AI_part_001.m4a']


In [13]:
#Transcribe the Chunks
from pathlib import Path
from openai import OpenAI

client = OpenAI()

CHUNKS_DIR = Path("./data/audio_chunks")
TRANSCRIPT_DIR = Path("./data/podcasts")
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)

chunk_files = sorted(CHUNKS_DIR.glob("*.m4a"))
assert chunk_files, f"No .m4a chunk files found in {CHUNKS_DIR.resolve()}"

# Group chunks by podcast base name (before "_part_")
podcast_groups = {}
for chunk in chunk_files:
    base_name = chunk.stem.split("_part_")[0]
    podcast_groups.setdefault(base_name, []).append(chunk)

print("Podcasts found from chunks:", list(podcast_groups.keys()))

for podcast_name, parts in podcast_groups.items():
    print(f"\n🎙️ Transcribing podcast: {podcast_name} ({len(parts)} parts)")
    full_text = ""

    for part in sorted(parts):
        print(f"  → {part.name}")
        with open(part, "rb") as f:
            tr = client.audio.transcriptions.create(
                model="whisper-1",
                file=f
            )
        full_text += tr.text.strip() + "\n"

    output_path = TRANSCRIPT_DIR / f"{podcast_name}.txt"
    with open(output_path, "w", encoding="utf-8") as out:
        out.write(full_text)

    print(f"✅ Saved transcript: {output_path}  ({len(full_text)} chars)")

Podcasts found from chunks: ['Red_Lines_AI_Act', 'The_Blueprint_For_Trustworthy_AI']

🎙️ Transcribing podcast: Red_Lines_AI_Act (3 parts)
  → Red_Lines_AI_Act_part_000.m4a
  → Red_Lines_AI_Act_part_001.m4a
  → Red_Lines_AI_Act_part_002.m4a
✅ Saved transcript: data\podcasts\Red_Lines_AI_Act.txt  (17069 chars)

🎙️ Transcribing podcast: The_Blueprint_For_Trustworthy_AI (2 parts)
  → The_Blueprint_For_Trustworthy_AI_part_000.m4a
  → The_Blueprint_For_Trustworthy_AI_part_001.m4a
✅ Saved transcript: data\podcasts\The_Blueprint_For_Trustworthy_AI.txt  (16406 chars)


In [ ]:
#Load (chunked) podcast transcripts and EU AI Act PDF 
from pathlib import Path
from langchain_community.document_loaders import TextLoader, PyPDFLoader

PODCAST_DIR = Path("./data/podcasts")
EU_ACT_PDF  = Path("./data/eu_ai_act.pdf")

docs = []

# EU AI Act
assert EU_ACT_PDF.exists(), f"EU AI Act PDF not found at: {EU_ACT_PDF.resolve()}"
pdf_loader = PyPDFLoader(str(EU_ACT_PDF))
eu_pages = pdf_loader.load()

for d in eu_pages:
    d.metadata.update({
        "source_type": "eu_ai_act",
        "source_id": EU_ACT_PDF.name,
        "title": "EU AI Act",
        "section": f"page_{d.metadata.get('page', 'NA')}",
    })
docs.extend(eu_pages)

# Podcast transcripts (txt)
txt_files = sorted(PODCAST_DIR.glob("*.txt"))
assert txt_files, f"No transcript .txt files found in: {PODCAST_DIR.resolve()}"

for fp in txt_files:
    loader = TextLoader(str(fp), encoding="utf-8")
    episode_docs = loader.load()
    for d in episode_docs:
        d.metadata.update({
            "source_type": "podcast",
            "source_id": fp.stem,
            "title": f"Trustworthy AI Podcast – {fp.stem}",
            "section": "transcript",
        })
    docs.extend(episode_docs)

print(f"✅ Loaded {len(docs)} documents total")
print("Sample metadata:", docs[0].metadata)
print("Sample text:", docs[0].page_content[:200].replace("\n"," "))

✅ Loaded 152 documents total
Sample metadata: {'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-02-26T14:23:21+00:00', 'title': 'EU AI Act', 'moddate': '2026-02-26T14:23:21+00:00', 'source': 'data\\eu_ai_act.pdf', 'total_pages': 150, 'page': 0, 'page_label': '1', 'source_type': 'eu_ai_act', 'source_id': 'eu_ai_act.pdf', 'section': 'page_0'}
Sample text: Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artiﬁcial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/20


In [16]:
#Chunk documents with metadata (source type, section, etc.)
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,      # chars (good starting point)
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for doc in docs:
    split_docs = splitter.split_documents([doc])
    for i, sd in enumerate(split_docs):
        sd.metadata["chunk_index"] = i  # index within that source document
        chunks.append(sd)

print("✅ Total chunks:", len(chunks))
print("Sample chunk metadata:", chunks[0].metadata)
print("Sample chunk text:", chunks[0].page_content[:250].replace("\n", " "))

✅ Total chunks: 656
Sample chunk metadata: {'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-02-26T14:23:21+00:00', 'title': 'EU AI Act', 'moddate': '2026-02-26T14:23:21+00:00', 'source': 'data\\eu_ai_act.pdf', 'total_pages': 150, 'page': 0, 'page_label': '1', 'source_type': 'eu_ai_act', 'source_id': 'eu_ai_act.pdf', 'section': 'page_0', 'chunk_index': 0}
Sample chunk text: Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artiﬁcial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/113


In [18]:
#Generate Embeddings for all chunks and create a Vectorshote (Chroma)
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_relevance_lab"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

print("✅ Vectorstore created.")
print("✅ Retriever ready (k=10).")

✅ Vectorstore created.
✅ Retriever ready (k=10).


In [21]:
# One baseline retrieval query (LangChain v0.3+ style)
query = "What are the obligations for providers of high-risk AI systems under the EU AI Act?"

results = retriever.invoke(query)

for i, r in enumerate(results[:5], 1):
    md = r.metadata
    print(f"\n#{i} | source_type={md.get('source_type')} | source_id={md.get('source_id')} | section={md.get('section')} | chunk_index={md.get('chunk_index')}")
    print(r.page_content[:350].replace("\n", " ") + ("..." if len(r.page_content) > 350 else ""))


#1 | source_type=eu_ai_act | source_id=eu_ai_act.pdf | section=page_86 | chunk_index=0
4.   For high-risk AI systems referred to in points 1, 6 and 7 of Annex III, in the areas of law enforcement, migration, asylum and border control management, the registration referred to in paragraphs 1, 2 and 3 of this Article shall be in a secure non-public section of the EU database referred to in Article 71 and shall include only the following...

#2 | source_type=eu_ai_act | source_id=eu_ai_act.pdf | section=page_68 | chunk_index=0
Article 21 Cooperation with competent authorities 1.   Providers of high-risk AI systems shall, upon a reasoned request by a competent authority, provide that authority all the information and documentation necessary to demonstrate the conformity of the high-risk AI system with the requirements set out in Section 2, in a language which can be easil...

#3 | source_type=eu_ai_act | source_id=eu_ai_act.pdf | section=page_25 | chunk_index=2
irrespective of whether they

In [23]:
#Only serac EU only Metadata filtering
# EU-only filtered retriever
eu_only_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 10,
        "filter": {"source_type": "eu_ai_act"}   # Chroma metadata filter
    }
)

query = "What are the obligations for providers of high-risk AI systems under the EU AI Act?"

eu_only_results = eu_only_retriever.invoke(query)

for i, r in enumerate(eu_only_results[:5], 1):
    md = r.metadata
    print(f"\n#{i} | source_type={md.get('source_type')} | section={md.get('section')} | chunk_index={md.get('chunk_index')}")
    print(r.page_content[:350].replace("\n", " ") + ("..." if len(r.page_content) > 350 else ""))


#1 | source_type=eu_ai_act | section=page_86 | chunk_index=0
4.   For high-risk AI systems referred to in points 1, 6 and 7 of Annex III, in the areas of law enforcement, migration, asylum and border control management, the registration referred to in paragraphs 1, 2 and 3 of this Article shall be in a secure non-public section of the EU database referred to in Article 71 and shall include only the following...

#2 | source_type=eu_ai_act | section=page_68 | chunk_index=0
Article 21 Cooperation with competent authorities 1.   Providers of high-risk AI systems shall, upon a reasoned request by a competent authority, provide that authority all the information and documentation necessary to demonstrate the conformity of the high-risk AI system with the requirements set out in Section 2, in a language which can be easil...

#3 | source_type=eu_ai_act | section=page_25 | chunk_index=2
irrespective of whether they may be used as high-risk AI systems as such by other providers or as componen

In [25]:
#Step 2 Generate Embeddings and Initial Retrieval (Baseline)
def show_top5(label, query, retriever_obj):
    print(f"\n======================")
    print(f"{label}")
    print(f"QUERY: {query}")
    print(f"======================")
    results = retriever_obj.invoke(query)
    for i, r in enumerate(results[:5], 1):
        md = r.metadata
        print(f"\n#{i} | source_type={md.get('source_type')} | source_id={md.get('source_id')} | section={md.get('section')} | chunk_index={md.get('chunk_index')}")
        print(r.page_content[:300].replace("\n", " ") + ("..." if len(r.page_content) > 300 else ""))

# Baseline retriever (already created): retriever = vectorstore.as_retriever(k=10)
q1 = "What are the obligations for providers of high-risk AI systems under the EU AI Act?"
q2 = "What does the podcast say about trustworthy AI and why it matters?"

show_top5("Baseline retrieval (EU AI Act query)", q1, retriever)
show_top5("Baseline retrieval (Podcast query)", q2, retriever)



Baseline retrieval (EU AI Act query)
QUERY: What are the obligations for providers of high-risk AI systems under the EU AI Act?

#1 | source_type=eu_ai_act | source_id=eu_ai_act.pdf | section=page_86 | chunk_index=0
4.   For high-risk AI systems referred to in points 1, 6 and 7 of Annex III, in the areas of law enforcement, migration, asylum and border control management, the registration referred to in paragraphs 1, 2 and 3 of this Article shall be in a secure non-public section of the EU database referred to ...

#2 | source_type=eu_ai_act | source_id=eu_ai_act.pdf | section=page_68 | chunk_index=0
Article 21 Cooperation with competent authorities 1.   Providers of high-risk AI systems shall, upon a reasoned request by a competent authority, provide that authority all the information and documentation necessary to demonstrate the conformity of the high-risk AI system with the requirements set ...

#3 | source_type=eu_ai_act | source_id=eu_ai_act.pdf | section=page_25 | chunk_index=2

In [28]:
#Baseline Results for Later Comparison
def pack_results(results, n=10):
    packed = []
    for r in results[:n]:
        md = r.metadata
        packed.append({
            "source_type": md.get("source_type"),
            "source_id": md.get("source_id"),
            "section": md.get("section"),
            "chunk_index": md.get("chunk_index"),
            "text_preview": r.page_content[:250].replace("\n", " ")
        })
    return packed

q1 = "What are the obligations for providers of high-risk AI systems under the EU AI Act?"
q2 = "What does the podcast say about trustworthy AI and why it matters?"

baseline_snapshots = {
    "EU_query_unfiltered": pack_results(retriever.invoke(q1), n=10),
    "EU_query_eu_only": pack_results(eu_retriever.invoke(q1), n=10),
    "Podcast_query_unfiltered": pack_results(retriever.invoke(q2), n=10),
    "Podcast_query_podcast_only": pack_results(podcast_retriever.invoke(q2), n=10),
}

# optional: write to json so you can include it in your report
import json
with open("baseline_retrieval_snapshots.json", "w", encoding="utf-8") as f:
    json.dump(baseline_snapshots, f, indent=2, ensure_ascii=False)

print("✅ Saved baseline snapshots to baseline_retrieval_snapshots.json")
print("Example item:", baseline_snapshots["EU_query_eu_only"][0])

✅ Saved baseline snapshots to baseline_retrieval_snapshots.json
Example item: {'source_type': 'eu_ai_act', 'source_id': 'eu_ai_act.pdf', 'section': 'page_86', 'chunk_index': 0, 'text_preview': '4.   For high-risk AI systems referred to in points 1, 6 and 7 of Annex III, in the areas of law enforcement, migration, asylum and border control management, the registration referred to in paragraphs 1, 2 and 3 of this Article shall be in a secure '}


In [30]:
#Step 3.1 — Retrieve candidates with vector similarity scores (baseline inputs for scoring)
# Step 3.1: get initial retrieved chunks WITH vector scores
# NOTE: Chroma returns a "distance" score (lower = better) for many configs.
query = "What are the obligations for providers of high-risk AI systems under the EU AI Act?"

# pull more candidates than final top_n (reranking happens later)
K = 30

# returns List[Tuple[Document, float]]
candidates_with_scores = vectorstore.similarity_search_with_score(
    query=query,
    k=K,
    filter={"source_type": "eu_ai_act"}  # keep EU-only for clean legal query
)

print("Candidates retrieved:", len(candidates_with_scores))
doc0, score0 = candidates_with_scores[0]
print("Top-1 raw score (distance-ish):", score0)
print("Top-1 metadata:", doc0.metadata)
print("Top-1 preview:", doc0.page_content[:250].replace("\n", " "))

Candidates retrieved: 30
Top-1 raw score (distance-ish): 0.47615668177604675
Top-1 metadata: {'total_pages': 150, 'moddate': '2026-02-26T14:23:21+00:00', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36', 'chunk_index': 0, 'title': 'EU AI Act', 'section': 'page_86', 'source': 'data\\eu_ai_act.pdf', 'source_id': 'eu_ai_act.pdf', 'page_label': '87', 'producer': 'Skia/PDF m145', 'page': 86, 'source_type': 'eu_ai_act', 'creationdate': '2026-02-26T14:23:21+00:00'}
Top-1 preview: 4.   For high-risk AI systems referred to in points 1, 6 and 7 of Annex III, in the areas of law enforcement, migration, asylum and border control management, the registration referred to in paragraphs 1, 2 and 3 of this Article shall be in a secure 


In [ ]:
#Use LLM to score relevance of each chunk - Test first with only 8 chunks
import re
from openai import OpenAI

client = OpenAI()

def llm_relevance_score(query: str, chunk: str) -> float:
    """
    Returns a relevance score in [0,10].
    We force the model to output ONLY a number to make parsing robust.
    """
    system = "You are a strict relevance scorer for retrieval in a legal RAG system."
    user = f"""Score how relevant the CHUNK is for answering the QUERY.

Rules:
- Output ONLY a number from 0 to 10 (can be decimal).
- 0 = completely irrelevant
- 10 = directly answers the query with specific obligations/details
- Prefer chunks that list provider obligations (requirements, duties, must/shall), not tangential mentions.

QUERY:
{query}

CHUNK:
{chunk[:2000]}
"""

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ],
        temperature=0
    )
    text = resp.choices[0].message.content.strip()

    # parse first number
    m = re.search(r"[-+]?\d+(\.\d+)?", text)
    if not m:
        raise ValueError(f"Could not parse numeric score from: {text}")
    score = float(m.group(0))
    return max(0.0, min(10.0, score))

# Score the first 8 candidates as a test (keeps cost low for now)
N_TEST = 8
scored_preview = []
for doc, sim_score in candidates_with_scores[:N_TEST]:
    rel = llm_relevance_score(query, doc.page_content)
    scored_preview.append((rel, sim_score, doc))

print("✅ Scored first", N_TEST, "chunks.")
for i, (rel, sim, doc) in enumerate(scored_preview, 1):
    md = doc.metadata
    print(f"\n#{i}  LLM_rel={rel:>4} | sim_score={sim:.3f} | section={md.get('section')} | chunk_index={md.get('chunk_index')}")
    print(doc.page_content[:220].replace("\n"," ") + "...")

✅ Scored first 8 chunks.

#1  LLM_rel= 6.5 | sim_score=0.476 | section=page_86 | chunk_index=0
4.   For high-risk AI systems referred to in points 1, 6 and 7 of Annex III, in the areas of law enforcement, migration, asylum and border control management, the registration referred to in paragraphs 1, 2 and 3 of this...

#2  LLM_rel= 8.0 | sim_score=0.499 | section=page_68 | chunk_index=0
Article 21 Cooperation with competent authorities 1.   Providers of high-risk AI systems shall, upon a reasoned request by a competent authority, provide that authority all the information and documentation necessary to ...

#3  LLM_rel= 6.0 | sim_score=0.509 | section=page_25 | chunk_index=2
irrespective of whether they may be used as high-risk AI systems as such by other providers or as components of high-risk AI systems and unless provided otherwise under this Regulation, closely cooperate with the provide...

#4  LLM_rel= 3.0 | sim_score=0.510 | section=page_25 | chunk_index=1
the AI system becomes a

In [32]:
#Score all 30 candidates and combine similiarity, LLM score and reorder
import numpy as np

# 1) Score all K candidates with the LLM
scored = []
for doc, dist in candidates_with_scores:
    rel = llm_relevance_score(query, doc.page_content)
    scored.append({"doc": doc, "dist": float(dist), "rel": float(rel)})

# 2) Normalize distance->similarity to [0,1] (higher better)
dists = np.array([x["dist"] for x in scored], dtype=float)
d_min, d_max = float(dists.min()), float(dists.max())
den = (d_max - d_min) if (d_max - d_min) > 1e-9 else 1.0

for x in scored:
    sim_norm = 1.0 - ((x["dist"] - d_min) / den)   # 1 = best (closest), 0 = worst
    rel_norm = x["rel"] / 10.0
    x["sim_norm"] = float(sim_norm)
    x["rel_norm"] = float(rel_norm)

# 3) Combine (weights you can tune)
W_SIM = 0.35
W_REL = 0.65

for x in scored:
    x["combined"] = W_SIM * x["sim_norm"] + W_REL * x["rel_norm"]

# 4) Sort by combined score descending
reranked = sorted(scored, key=lambda z: z["combined"], reverse=True)

print("✅ LLM relevance scoring rerank complete.")
print(f"Distance range: min={d_min:.3f}, max={d_max:.3f}")
print(f"Weights: W_SIM={W_SIM}, W_REL={W_REL}")

# 5) Show top 8 after combining
for i, x in enumerate(reranked[:8], 1):
    md = x["doc"].metadata
    print(f"\n#{i} combined={x['combined']:.3f} | rel={x['rel']:.1f} | sim_norm={x['sim_norm']:.3f} | section={md.get('section')} | chunk_index={md.get('chunk_index')}")
    print(x["doc"].page_content[:260].replace("\n"," ") + "...")

✅ LLM relevance scoring rerank complete.
Distance range: min=0.476, max=0.601
Weights: W_SIM=0.35, W_REL=0.65

#1 combined=0.809 | rel=9.0 | sim_norm=0.640 | section=page_65 | chunk_index=1
The technical solutions aiming to ensure the cybersecurity of high-risk AI systems shall be appropriate to the relevant circumstances and the risks. The technical solutions to address AI specific vulnerabilities shall include, where appropriate, measures to pr...

#2 combined=0.805 | rel=8.0 | sim_norm=0.816 | section=page_68 | chunk_index=0
Article 21 Cooperation with competent authorities 1.   Providers of high-risk AI systems shall, upon a reasoned request by a competent authority, provide that authority all the information and documentation necessary to demonstrate the conformity of the high-r...

#3 combined=0.786 | rel=9.0 | sim_norm=0.574 | section=page_71 | chunk_index=2
Article 26 Obligations of deployers of high-risk AI systems 1.   Deployers of high-risk AI systems shall take appropriate 

In [33]:
#Save LLM relevance scoring results for later comparison
def pack_reranked(reranked_list, n=10):
    packed = []
    for x in reranked_list[:n]:
        doc = x["doc"]
        md = doc.metadata
        packed.append({
            "combined": round(x["combined"], 4),
            "llm_rel": x["rel"],
            "sim_norm": round(x["sim_norm"], 4),
            "source_type": md.get("source_type"),
            "source_id": md.get("source_id"),
            "section": md.get("section"),
            "chunk_index": md.get("chunk_index"),
            "text_preview": doc.page_content[:250].replace("\n", " "),
        })
    return packed

llm_scoring_snapshots = {
    "query": query,
    "k_candidates": K,
    "weights": {"W_SIM": W_SIM, "W_REL": W_REL},
    "top10": pack_reranked(reranked, n=10),
}

import json
with open("llm_relevance_scoring_snapshots.json", "w", encoding="utf-8") as f:
    json.dump(llm_scoring_snapshots, f, indent=2, ensure_ascii=False)

print("✅ Saved LLM scoring snapshots to llm_relevance_scoring_snapshots.json")
print("Top-1 snapshot:", llm_scoring_snapshots["top10"][0])

✅ Saved LLM scoring snapshots to llm_relevance_scoring_snapshots.json
Top-1 snapshot: {'combined': 0.8091, 'llm_rel': 9.0, 'sim_norm': 0.6404, 'source_type': 'eu_ai_act', 'source_id': 'eu_ai_act.pdf', 'section': 'page_65', 'chunk_index': 1, 'text_preview': 'The technical solutions aiming to ensure the cybersecurity of high-risk AI systems shall be appropriate to the relevant circumstances and the risks. The technical solutions to address AI specific vulnerabilities shall include, where appropriate, meas'}


In [37]:
#Create Cohere Rerank Retriever
import cohere
import os

assert "COHERE_API_KEY" in os.environ, "COHERE_API_KEY not found in env (.env)."
co = cohere.Client(os.environ["COHERE_API_KEY"])

# Use the SAME candidates as Step 3 for a fair comparison
# candidates_with_scores is List[(Document, dist)] from your Step 3.1 cell
docs = [doc.page_content for doc, _ in candidates_with_scores]

cohere_rerank = co.rerank(
    model="rerank-v3.5",
    query=query,
    documents=docs,
    top_n=10
)

# Build reranked list (keep doc + vector dist + cohere relevance)
cohere_reranked = []
for r in cohere_rerank.results:
    idx = r.index
    doc, dist = candidates_with_scores[idx]
    cohere_reranked.append({
        "cohere_score": float(r.relevance_score),
        "dist": float(dist),
        "doc": doc
    })

print("✅ Cohere rerank complete. Top 5:")
for i, x in enumerate(cohere_reranked[:5], 1):
    md = x["doc"].metadata
    print(f"\n#{i} cohere_score={x['cohere_score']:.4f} | dist={x['dist']:.3f} | section={md.get('section')} | chunk_index={md.get('chunk_index')}")
    print(x["doc"].page_content[:260].replace("\n"," ") + "...")

✅ Cohere rerank complete. Top 5:

#1 cohere_score=0.9360 | dist=0.521 | section=page_65 | chunk_index=1
The technical solutions aiming to ensure the cybersecurity of high-risk AI systems shall be appropriate to the relevant circumstances and the risks. The technical solutions to address AI specific vulnerabilities shall include, where appropriate, measures to pr...

#2 cohere_score=0.9123 | dist=0.499 | section=page_68 | chunk_index=0
Article 21 Cooperation with competent authorities 1.   Providers of high-risk AI systems shall, upon a reasoned request by a competent authority, provide that authority all the information and documentation necessary to demonstrate the conformity of the high-r...

#3 cohere_score=0.9097 | dist=0.536 | section=page_70 | chunk_index=2
a high-risk AI system for the purposes of this Regulation and shall be subject to the obligations of the provider under Article 16, in any of the following circumstances: (a) they put their name or trademark on a high-risk AI 

In [38]:
#Save Cohere rerank snapshots
import json

def pack_cohere(cohere_reranked_list, n=10):
    packed = []
    for x in cohere_reranked_list[:n]:
        doc = x["doc"]
        md = doc.metadata
        packed.append({
            "cohere_score": round(x["cohere_score"], 4),
            "dist": round(x["dist"], 6),
            "source_type": md.get("source_type"),
            "source_id": md.get("source_id"),
            "section": md.get("section"),
            "chunk_index": md.get("chunk_index"),
            "text_preview": doc.page_content[:250].replace("\n", " "),
        })
    return packed

cohere_rerank_snapshots = {
    "query": query,
    "k_candidates": K,
    "top10": pack_cohere(cohere_reranked, n=10),
}

with open("cohere_rerank_snapshots.json", "w", encoding="utf-8") as f:
    json.dump(cohere_rerank_snapshots, f, indent=2, ensure_ascii=False)

print("✅ Saved Cohere rerank snapshots to cohere_rerank_snapshots.json")
print("Top-1 snapshot:", cohere_rerank_snapshots["top10"][0])

✅ Saved Cohere rerank snapshots to cohere_rerank_snapshots.json
Top-1 snapshot: {'cohere_score': 0.936, 'dist': 0.521203, 'source_type': 'eu_ai_act', 'source_id': 'eu_ai_act.pdf', 'section': 'page_65', 'chunk_index': 1, 'text_preview': 'The technical solutions aiming to ensure the cybersecurity of high-risk AI systems shall be appropriate to the relevant circumstances and the risks. The technical solutions to address AI specific vulnerabilities shall include, where appropriate, meas'}


In [39]:
#Compare LLM scoring with Cohere
def top_ids_from_llm(reranked_llm, n=10):
    out = []
    for x in reranked_llm[:n]:
        md = x["doc"].metadata
        out.append((md.get("section"), md.get("chunk_index")))
    return out

def top_ids_from_cohere(cohere_reranked, n=10):
    out = []
    for x in cohere_reranked[:n]:
        md = x["doc"].metadata
        out.append((md.get("section"), md.get("chunk_index")))
    return out

llm_top10 = top_ids_from_llm(reranked, 10)
cohere_top10 = top_ids_from_cohere(cohere_reranked, 10)

print("Top 10 (LLM relevance scoring) vs Top 10 (Cohere rerank)")
for i in range(10):
    l = llm_top10[i] if i < len(llm_top10) else None
    c = cohere_top10[i] if i < len(cohere_top10) else None
    print(f"{i+1:>2}. LLM={l}   |   Cohere={c}")

Top 10 (LLM relevance scoring) vs Top 10 (Cohere rerank)
 1. LLM=('page_65', 1)   |   Cohere=('page_65', 1)
 2. LLM=('page_68', 0)   |   Cohere=('page_68', 0)
 3. LLM=('page_71', 2)   |   Cohere=('page_70', 2)
 4. LLM=('page_86', 0)   |   Cohere=('page_66', 2)
 5. LLM=('page_25', 0)   |   Cohere=('page_71', 0)
 6. LLM=('page_59', 1)   |   Cohere=('page_71', 1)
 7. LLM=('page_42', 1)   |   Cohere=('page_71', 2)
 8. LLM=('page_70', 2)   |   Cohere=('page_86', 0)
 9. LLM=('page_70', 1)   |   Cohere=('page_25', 0)
10. LLM=('page_71', 0)   |   Cohere=('page_42', 1)


In [40]:
#Step 5 Add Metadata Filtering - metadata-filtered retrievers (baseline, no reranking yet)

eu_only_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10, "filter": {"source_type": "eu_ai_act"}}
)

podcast_only_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10, "filter": {"source_type": "podcast"}}
)

print("✅ Step 5.1 done: created eu_only_retriever and podcast_only_retriever")

✅ Step 5.1 done: created eu_only_retriever and podcast_only_retriever


In [41]:
#Demonstrate Metadata filtering with an ambiguous query
ambiguous_query = "What is meant by trustworthy AI and what requirements are mentioned?"

show_top5("Unfiltered baseline", ambiguous_query, retriever)
show_top5("EU-only baseline (metadata filter)", ambiguous_query, eu_only_retriever)
show_top5("Podcast-only baseline (metadata filter)", ambiguous_query, podcast_only_retriever)


Unfiltered baseline
QUERY: What is meant by trustworthy AI and what requirements are mentioned?

#1 | source_type=podcast | source_id=The_Blueprint_For_Trustworthy_AI | section=transcript | chunk_index=0
So imagine for a second you're driving across, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No. You just, you trust it. You just drive. You trust the bridge. You trust the e...

#2 | source_type=podcast | source_id=The_Blueprint_For_Trustworthy_AI | section=transcript | chunk_index=2
. And a checklist that every developer should probably have tattooed on their arm. Okay, let's start with the big picture. The source material defines trustworthy AI as having three components. Right. And they have to exist through the entire lifecycle of the system. Not just at launch. Not just at ...

#3 | source_type=podcast | source_id=The_Blueprint_For_Trustworthy_AI | sectio

In [42]:
#Print only the source_type + source_id + section for each setting
def show_sources_only(label, query, retriever_obj, n=5):
    print(f"\n--- {label} ---")
    res = retriever_obj.invoke(query)
    for i, r in enumerate(res[:n], 1):
        md = r.metadata
        print(f"{i}. {md.get('source_type')} | {md.get('source_id')} | {md.get('section')} | chunk={md.get('chunk_index')}")

ambiguous_query = "What is meant by trustworthy AI and what requirements are mentioned?"

show_sources_only("Unfiltered", ambiguous_query, retriever)
show_sources_only("EU-only", ambiguous_query, eu_only_retriever)
show_sources_only("Podcast-only", ambiguous_query, podcast_only_retriever)


--- Unfiltered ---
1. podcast | The_Blueprint_For_Trustworthy_AI | transcript | chunk=0
2. podcast | The_Blueprint_For_Trustworthy_AI | transcript | chunk=2
3. podcast | The_Blueprint_For_Trustworthy_AI | transcript | chunk=16
4. eu_ai_act | eu_ai_act.pdf | page_23 | chunk=0
5. eu_ai_act | eu_ai_act.pdf | page_8 | chunk=0

--- EU-only ---
1. eu_ai_act | eu_ai_act.pdf | page_23 | chunk=0
2. eu_ai_act | eu_ai_act.pdf | page_8 | chunk=0
3. eu_ai_act | eu_ai_act.pdf | page_64 | chunk=3
4. eu_ai_act | eu_ai_act.pdf | page_36 | chunk=3
5. eu_ai_act | eu_ai_act.pdf | page_65 | chunk=0

--- Podcast-only ---
1. podcast | The_Blueprint_For_Trustworthy_AI | transcript | chunk=0
2. podcast | The_Blueprint_For_Trustworthy_AI | transcript | chunk=2
3. podcast | The_Blueprint_For_Trustworthy_AI | transcript | chunk=16
4. podcast | The_Blueprint_For_Trustworthy_AI | transcript | chunk=1
5. podcast | The_Blueprint_For_Trustworthy_AI | transcript | chunk=9


In [43]:
#Step 6 Complete RAG Pipeline with Reranking
from openai import OpenAI

client = OpenAI()

def answer_with_rag(query, retriever_obj):
    docs = retriever_obj.invoke(query)
    
    context = "\n\n".join(
        f"[Source: {d.metadata.get('section')}]\n{d.page_content}"
        for d in docs
    )
    
    prompt = f"""You are a legal assistant.
Answer the question using ONLY the provided context.
If the answer is not in the context, say so.

QUESTION:
{query}

CONTEXT:
{context}
"""
    
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    return resp.choices[0].message.content

In [45]:
# Step 6.2: RAG with baseline vs Cohere reranking (only docs selection changes)

query = "What are the obligations for providers of high-risk AI systems under the EU AI Act?"

# --- Baseline docs (vector search only) ---
baseline_docs = [doc for doc, _dist in candidates_with_scores[:10]]  # top 10 from Step 3.1 (EU-only)

# --- Cohere reranked docs (same candidates, reranked to top 10) ---
cohere_docs = [x["doc"] for x in cohere_reranked[:10]]

print("\n========== BASELINE RAG ANSWER ==========")
print(rag_answer(query, baseline_docs))

print("\n========== COHERE-RERANKED RAG ANSWER ==========")
print(rag_answer(query, cohere_docs))


========== BASELINE RAG ANSWER ==========
Providers of high-risk AI systems under the EU AI Act have the following obligations:

1. **Compliance**: They must ensure that their high-risk AI systems comply with the requirements set out in Section 2 of the Act.

2. **Identification**: They must indicate their name, registered trade name or trademark, and contact address on the high-risk AI system, its packaging, or accompanying documentation, as applicable.

3. **Quality Management System**: They are required to have a quality management system in place that complies with Article 17.

4. **Cooperation with Authorities**: Upon a reasoned request from a competent authority, providers must provide all necessary information and documentation to demonstrate conformity with the requirements. They must also grant access to automatically generated logs of the high-risk AI system, as applicable.

5. **Registration**: High-risk AI systems must be registered in a secure non-public section of the EU

In [46]:
#Manual Evaluation
eval_template = [
    {
        "query": query,
        "baseline_notes": "",
        "cohere_notes": "",
        "winner_for_precision": "",
        "winner_for_coverage": "",
        "overall_winner": "",
    }
]

import pandas as pd
pd.DataFrame(eval_template)

,query,baseline_notes,cohere_notes,winner_for_precision,winner_for_coverage,overall_winner
0,What are the obligations for providers of high...,,,,,
